# Camera projection and depth

Distinguish camera z-depth from range and evaluate explicit validity masks.

Run with the repository virtual environment. These are synthetic research experiments; units and assumptions are explicit below.

## Camera convention

Camera x points right, y down, z forward. These analytical intrinsics are synthetic.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from autonomous_vehicles.evaluation.depth import depth_metrics, median_scale
from autonomous_vehicles.simulation.camera import (
    CameraIntrinsics,
    depth_to_points,
    project_points,
    unproject_pixels,
)

camera = CameraIntrinsics(width=320, height=180, fx=200, fy=200, cx=159.5, cy=89.5)
print(camera.matrix)

## Project a point grid

Points behind the near plane or outside the image are marked invisible.

In [ ]:
x, y = np.meshgrid(np.linspace(-4, 4, 9), np.linspace(-2, 2, 5))
points = np.column_stack((x.ravel(), y.ravel(), np.full(x.size, 10)))
projection = project_points(points, camera)
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(projection.pixels[:, 0], projection.pixels[:, 1], c=points[:, 0])
ax.set_xlim(0, camera.width)
ax.set_ylim(camera.height, 0)
ax.set_xlabel("Pixel u")
ax.set_ylabel("Pixel v")
plt.show()

## Unproject

Depth is the z coordinate. Off-axis Euclidean distance is larger.

In [ ]:
restored = unproject_pixels(projection.pixels, projection.depth, camera)
np.testing.assert_allclose(restored, points, atol=1e-12)
ranges = np.linalg.norm(points, axis=1)
print("Depth min/max:", projection.depth.min(), projection.depth.max())
print("Range min/max:", ranges.min(), ranges.max())
assert ranges.max() > projection.depth.max()

## Depth validity

Invalid targets are excluded. Invalid predictions at valid target pixels are errors.

In [ ]:
rows, columns = np.mgrid[: camera.height, : camera.width]
target = 5 + rows / camera.height * 20
target = target.astype(float)
target[:20] = np.nan
rng = np.random.default_rng(42)
prediction = np.nan_to_num(target, nan=1) + rng.normal(0, 0.2, target.shape)
metrics = depth_metrics(prediction, target)
print(metrics)
cloud = depth_to_points(target, camera, stride=12)
print("Point cloud shape:", cloud.shape)

## Scale ambiguity

Median alignment changes the evaluation contract. Report both raw and aligned values when used.

In [ ]:
scaled_prediction = np.nan_to_num(target, nan=1) * 0.7
scale = median_scale(scaled_prediction, target)
raw = depth_metrics(scaled_prediction, target)
aligned = depth_metrics(scaled_prediction * scale, target)
print(
    {
        "fitted_scale": scale,
        "raw_mae_m": raw.mean_absolute_error,
        "aligned_mae_m": aligned.mean_absolute_error,
    }
)
fig, ax = plt.subplots(figsize=(7, 3))
view = ax.imshow(target, cmap="viridis")
fig.colorbar(view, ax=ax, label="Depth (m)")
plt.show()

## Research record

Record the seed, configuration, dependency lock and observed failures with each experiment. Results here demonstrate the implemented model under its stated assumptions. They do not establish performance outside this synthetic distribution.